# 词元和嵌入

词元和嵌入是使用LLM的两个核心概念。正如我们在第1章所见，它们对理解语言人工智能的历史至关重要，而且如图2-1所示，如果没有对词元和嵌入的深入理解，我们就无法清楚地了解LLM的工作原理、构建方式及其未来的发展方向。

<div align="center">
    <img src="./picture/2-1.jpg">
    <p>图2-1：语言模型处理文本时会将其分成小块，称为词元。为了理解自然语言，语言模型需要将词元转换为数值表示，即嵌入向量</p>
</div>

本章我们将深入探讨词元的本质以及LLM使用的分词方法。然后，我们将探讨著名的word2vec嵌入方法，它是现代LLM的先驱。我们将了解word2vec如何扩展词元嵌入(tokenembedding)的概念，来构建商业推荐系统，我们使用的许多互联网应用都是由推荐系统支持的。最后，我们将从词元嵌入过渡到句子或文本嵌入，即整个句子或文档可以用一个向量来表示——这为本书第二部分将要介绍的语义搜索和主题建模等应用奠定了基础。

## 2.1 LLM的分词

在撰写本书时，大多数人与语言模型交互的方式是通过网页平台，它提供用户与语言模型之间的聊天界面。你可能会注意到，模型并不是一次性生成所有输出，而是一次生成一个词元。

词元不仅是模型的输出单位，也是模型查看输入的方式。发送给模型的提示词首先被分解成词元，我们接下来将对此进行讨论。

### 2.1.1 分词器如何处理语言模型的输入

从外部来看，生成式LLM接收输入提示词并生成响应，如图2-2所示。

<div align="center">
    <img src="./picture/2-2.jpg">
    <p>图2-2：语言模型及其输入提示词的架构图</p>
</div>

然而，在将提示词呈现给语言模型之前，它首先要通过分词器将其分解成片段。你可以在OpenAI平台上找到GPT-4分词器的示例。在文本框中输入文本，它会显示如图2-3中的输出，其中词元以不同的颜色显示。

让我们看一个代码示例，并亲自与这些词元进行交互。在这里，我们将下载一个LLM，并了解如何在使用LLM生成文本之前对输入进行分词。

<div align="center">
    <img src="./picture/2-3.jpg">
    <p>图2-3：在模型处理文本之前，分词器会将文本分解成词或子词。这是根据特定的方法和训练过程进行的（图片来源：OpenAI平台官网）</p>
</div>

### 2.1.2 下载和运行LLM

让我们首先像第1章那样，加载模型及其分词器：

In [10]:
from transformers import AutoModelForCausalLM, AutoTokenizer
# 加载模型及其分词器
model = AutoModelForCausalLM.from_pretrained(
    "../models/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
)
tokenizer = AutoTokenizer.from_pretrained("../models/Phi-3-mini-4k-instruct")


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

然后我们就能进行实际的生成了。首先声明提示词，然后对其进行分词，再将这些词元传递给模型，模型随后生成输出。在这个例子中，我们要求模型只生成20个新词元：

In [12]:
import torch
prompt = """Write an email apologizing to Sarah for the tragic gardening mishap.
Explain how it happened.<|assistant|>"""
# 对输入提示词进行分词
input_ids = tokenizer(prompt, return_tensors="pt").input_ids

# 手动在开头加上 <s>（BOS token）
bos_id = tokenizer.bos_token_id  # Phi-3 的 <s> 对应的 id 是 1（32001 是 <|assistant|>）
bos_tensor = torch.tensor([[bos_id]], dtype=input_ids.dtype)
input_ids = torch.cat([bos_tensor, input_ids], dim=1).to("cuda")
# 生成文本
generation_output = model.generate(
    input_ids=input_ids,
    max_new_tokens=20
)
# 打印输出
print(tokenizer.decode(generation_output[0], skip_special_tokens=False))

<s> Write an email apologizing to Sarah for the tragic gardening mishap.
Explain how it happened.<|assistant|> Subject: Sincere Apologies for the Gardening Mishap

Dear Sarah


粗体文本是模型生成的20个词元。

从代码中可以看到，模型实际上并没有直接处理提示词文本。相反，输入提示词是由分词器处理的。分词器在变量input_ids中返回了模型所需的信息，随后模型将其用作输入。

In [13]:
print(input_ids)

tensor([[    1, 14350,   385,  4876, 27746,  5281,   304, 19235,   363,   278,
         25305,   293, 16423,   292,   286,   728,   481, 29889,    13,  9544,
          7420,   920,   372,  9559, 29889, 32001]], device='cuda:0')


这就是LLM输出的响应，如图2-4所示的一系列整数。每个整数都是特定词元（字符、词或词的一部分）的唯一ID。这些ID是分词器内部的一张词元表的索引，该表包含了分词器能够识别的所有词元。

<div align="center">
    <img src="./picture/2-4.jpg">
    <p>图2-4：分词器处理输入提示词，得到词元ID列表，这就是语言模型的实际输入。图中显示的具体词元ID仅作示例</p>
</div>

如果想查看这些ID，可以使用分词器的decode方法将ID转换回人类可阅读的文本：

In [14]:
for id in input_ids[0]:
    print(tokenizer.decode(id, skip_special_tokens=False))

<s>
Write
an
email
apolog
izing
to
Sarah
for
the
trag
ic
garden
ing
m
ish
ap
.


Exp
lain
how
it
happened
.
<|assistant|>


这就是分词器分解输入提示词的过程。需要注意以下几点：

· 第一个词元是ID 1(`<s>`)，这是一个表示文本开始的特殊词元；

· 一些词元是完整的单词（例如Write、an、email）​；

· 一些词元是单词的部分（例如apolog、izing、trag、ic）​；

· 标点符号是独立的词元。

注意空格字符不用单独的词元表示，代表词的一部分的词元（如izing和ic）在开头有一个特殊的隐藏字符，表示它们与文本中前面的词元相连。没有这个特殊字符的词元前面则都被视为有一个空格。

在输出端，我们也可以通过打印generation_output变量来查看模型生成的词元。打印的内容同时包括输入词元和输出词元（我们将新词元用粗体突出显示）​：

In [15]:
print(generation_output)

tensor([[    1, 14350,   385,  4876, 27746,  5281,   304, 19235,   363,   278,
         25305,   293, 16423,   292,   286,   728,   481, 29889,    13,  9544,
          7420,   920,   372,  9559, 29889, 32001,  3323,   622, 29901,   317,
          3742,   406,  6225, 11763,   363,   278, 19906,   292,   341,   728,
           481,    13,    13, 29928,   799, 19235]], device='cuda:0')


在上述例子中，模型生成了词元3323(Sub)，接着是词元622(ject)。它们一起组成了Subject这个词。然后是词元29901，即冒号，等等。就像输入端一样，我们需要分词器在输出端将词元ID转换为实际文本。我们使用分词器的decode方法来实现这一点。我们可以传入单个词元ID或它们的列表：

In [16]:
print(tokenizer.decode(3323))
print(tokenizer.decode(622))
print(tokenizer.decode([3323, 622]))
print(tokenizer.decode(29901))

Sub
ject
Subject
:


### 2.1.3 分词器如何分解文本

决定分词器如何分解输入提示词的因素主要有三个。

首先，在模型设计时，模型创建者会选择一种分词方法。流行的方法包括字节对编码（BPE，byte pair encoding，广泛用于GPT模型）和WordPiece（用于BERT模型）​。这些方法的相似之处在于，它们都旨在找到一组尽可能高效的词元来表示文本数据集，但它们采用不同的方式实现这一目标。

其次，在选择方法之后，我们需要做出一些分词器设计选择，如词表大小和使用哪些特殊词元。更多内容请参见2.1.5节。

最后，分词器需要在特定数据集上进行训练，以建立能最好地表示该数据集的词表。即使我们设置相同的方法和参数，在英语文本数据集上训练的分词器也会与在代码数据集或多语言文本数据集上训练的分词器不同。

除了把输入文本处理成语言模型的输入外，分词器还负责处理语言模型的输出，将生成的词元ID转换为与之关联的输出词或词元，如图2-5所示。

<div align="center">
    <img src="./picture/2-5.jpg">
    <p>图2-5：分词器还负责处理模型的输出，将输出词元ID转换为与该ID关联的词或词元</p>
</div>

### 2.1.4 词级、子词级、字符级与字节级分词

我们刚才讨论的分词方案被称为子词级分词(subwordtokenization)。这是最常用的分词方案，但并非唯一的方案。图2-6展示了四种主要的分词方式。

词级分词

 这种方法在早期的word2vec等模型中很常见，但在NLP中的使用越来越少。不过，由于其实用性，它在推荐系统等NLP以外的场景中得以应用，我们将在本章后面详细介绍。词级分词的一个挑战是，分词器可能无法处理分词器训练完成之后才出现在数据集中的新词。这也导致词表中存在大量仅有细微差别的词元（如apology、apologize、apologetic、apologist）​。这个问题可以通过子词级分词来解决，因为它有一个apolog词元，还有后缀词元（如-y、-ize、-etic、-ist）​，这些后缀词元与许多其他词元共用，从而形成更具表达能力的词表。

<div align="center">
    <img src="./picture/2-6.jpg">
    <p>图2-6：四种分词方法将文本分解成不同大小的词元（词级、子词级、字符级和字节级）</p>
</div>

子词级分词

这种方法包含词和词的一部分。除了前面提到的词表更具表达能力外，这种分词方法的另一个优势是能够表示新词，因为它可以将新词元分解成较小的字符，这些字符通常都在词表中。

字符级分词

这是另一种能够成功处理新词的分词方法，因为它仅仅依赖最原始的字母。虽然这使得分词更容易，但建模却变得更困难。在子词级分词中，模型可以用一个词元表示play，而使用字符级分词的模型则需要建模拼写出p-l-a-y的信息，此外还要建模序列的其他部分。

相比字符级分词，子词级分词可以在Transformer模型有限的上下文长度内，容纳更多文本。因此，在上下文长度为1024的模型中，使用子词级分词可以容纳字符级分词约三倍的文本（对于子词级词元，平均每个词元包含三个字符）​。

字节级分词

还有一种分词方法是将词元分解为表示unicode（统一码）字符的单个字节。像“CANINE: Pre-training an EfficientTokenization-Free Encoder for LanguageRepresentation”这样的论文概述了这类方法，它们也被称为“免分词编码”​。其他研究成果如“ByT5: Towards a Token-Free Future with Pre-trained Byte-to-Byte Models”表明，字节级分词是一种颇具竞争力的方法，尤其是在多语言场景中。

这里需要强调一个区别：某些子词分词器也会在其词表中将字节作为词元，在遇到无法用其他方式表示的字符时，这是最终备选方案，例如GPT-2和RoBERTa分词器就是这样做的。但这并不意味着它们是无分词的字节级分词器，因为它们并不是用字节来表示所有内容，而只是表示一部分内容，我们将在下一节中了解这一点。

如果你想深入了解分词器，推荐阅读Designing LargeLanguage Model Applications一书，其中对这一主题做了更详细的讨论。

### 2.1.5 比较训练好的LLM分词器

我们之前提到，分词器中出现的词元是由三个主要因素决定的：分词方法、用于初始化分词器的参数和特殊词元，以及用于训练分词器的数据集。接下来，我们对多个实际训练好的分词器进行比较，看看这些因素如何影响它们的行为。我们会看到较新的分词器是如何改变其行为以提升模型性能的，还会看到专门的模型（如代码生成模型）通常需要专门的分词器。

我们使用多个分词器来编码以下文本：

In [17]:
text = """
English and CAPITALIZATION
🎶鸟
show_tokens False None elif == >= else: two tabs:"  " Three tabs: "   “
12.0*50=600

"""


   由此，我们能够看到每个分词器是如何处理不同类型的词元的：

· 大小写· 英语以外的语言

· 表情符号(emoji)

· 编程代码，包括关键字和经常用于缩进的空白字符（例如在Python等语言中）

· 数字· 特殊词元。这类词元具有特定的作用，而不仅仅表示文本。它们包括表示文本开始或结束的词元（模型用结束词元来向系统表明已完成生成）​，以及我们随后将看到的其他功能性词元

我们按照从旧到新的时间顺序考察不同的分词器，看看它们如何对这段文本进行分词，以及这可能反映出语言模型的哪些特点。我们将使用以下函数对文本进行分词，并用彩色背景显示每个词元：

In [18]:
colors_list = [
    '102;194;165', '252;141;98', '141;160;203',
    '231;138;195', '166;216;84', '255;217;47'
]
def show_tokens(sentence, tokenizer_name):
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    token_ids = tokenizer(sentence).input_ids
    for idx, t in enumerate(token_ids):
        print(
            f'\x1b[0;30;48;2;{colors_list[idx % len(colors_list)]}m' +
            tokenizer.decode(t) +
            '\x1b[0m',
            end=' '
        )

#### BERT基座模型（大小写不敏感）(2018)


分词方法：WordPiece，参见论文“Japanese and Korean Voice Search”​。

词表大小：30522。

特殊词元：

unk_token [UNK]（未知词元，当分词器没有为某类字符进行特定编码时使用。）

sep_token [SEP]（分隔符词元，用于支持需要为模型提供两段文本的特定任务[在这些情况下，模型被称为交叉编码器(cross-encoder)]​。例如，我们将在第8章看到分隔符在重排序中的应用。）

pad_token [PAD]（填充词元，用于填充模型输入中未使用的位置，因为模型通常要求输入数据具有固定的长度（也就是其上下文窗口））

cls_token [CLS]（分类词元，主要用于分类任务的特殊词元，我们将在第4章看到。）


mask_token [MASK]（掩码词元，在训练过程中用于隐藏词元。）

分词后的文本：

<img src="./picture/bert-base.png">

BERT分词器有两个版本：大小写敏感（保留大写字母）的版本和大小写不敏感（所有大写字母先转换为小写字母）的版本。对于大小写不敏感（也是更受欢迎）的BERT分词器版本，我们注意到以下特点。

· 换行符消失了，这使得模型无法识别通过换行符体现的信息（例如，每一轮对话都位于新的一行的聊天记录）​。

· 所有文本都变成小写。

· capitalization这个词被编码为两个子词元：capital ##ization。##符号用来表示这个词元是与前面的词元相连的部分词元。这也是一种表示空格位置的方法，因为分词中约定没有##前缀的词元前面应该有一个空格。

· 表情符号和中文字符消失了，被替换成了[UNK]特殊词元，即“未知词元”​。

#### BERT基座模型（大小写敏感）(2018)


分词方法：WordPiece。

特殊词元：与大小写不敏感版本相同

分词后的文本：

<img src="./picture/bert.png">

大小写敏感版本的BERT分词器的主要不同之处在于包含了大写词元。

· 注意CAPITALIZATION现在被表示为八个词元：[插图]。

· 两种BERT分词器都会在输入文本前后分别添加一个起始词元[CLS]和结束词元[SEP]。​[CLS] 和[SEP] 是用于包裹输入文本的功能性词元，各有其用途。​[CLS] 代表分类(classification)，因为它有时被用于句子分类。​[SEP] 代表分隔符(separator)，用于在某些需要向模型传递两个句子的应用中分隔句子（例如，在第8章中，我们将使用[SEP] 词元来分隔查询文本和候选结果）​。

#### GPT-2(2019)


分词方法：BPE，参见论文“Neural Machine Translation ofRare Words with Subword Units”​。

词表大小：50257。

特殊词元：<|endoftext|>

分词后的文本：

<img src="./picture/gpt-2.png">

使用GPT-2分词器，我们注意到以下特点。

· 分词器保留了换行符。

· 保留了大小写，CAPITALIZATION这个词被表示为四个词元。

· 表情符号和中文字符“🎶鸟”​，分别被表示为多个词元。虽然我们看到这些词元显示为❓字符，但它们实际上代表不同的词元。例如，🎶这个表情符号被分解成词元ID为8582、236和113的词元。分词器能够成功地从这些词元中重构出原始字符。我们可以通过打印tokenizer.decode([8582, 236, 113])来验证，它会输出🎶。

· 两个制表符(tab)被表示为两个词元（在词表中的词元ID为197）​，三个制表符被表示为三个词元（词元号为220）​，最后一个空格被包含在表示闭合引号的词元中。

空白字符有什么意义？这类字符对于模型理解或生成代码非常重要。一个能够使用单个词元来表示连续四个空白字符的模型，更适合处理Python代码数据集。虽然模型也可以将其表示为四个不同的词元，但这会增加建模的难度，因为模型需要追踪缩进级别，这通常会导致性能下降。这个例子说明，合理选择分词粒度，可以帮助模型在特定任务上取得更好的表现。

#### FLAN-T5(2022)

分词方法：SentencePiece，参见论文“SentencePiece: Asimple and language independent subword tokenizerand detokenizer for Neural Text Processing”​。它支持BPE和一元语言模型（unigram language model，参见论文“Subword Regularization: Improving Neural NetworkTranslation Models with Multiple SubwordCandidates”​）​。

词表大小：32100。

特殊词元：

· unk_token `<unk>`（未知词元）

· pad_token `<pad>`（填充词元）

分词后的文本：

<img src="picture/FLAN-T5.jpg">

FLAN-T5系列模型使用SentencePiece方法。我们注意到以下两点：

· 该方法中没有换行符或空白字符词元，这会使模型处理代码变得具有挑战性；

· 表情符号和中文字符都被替换为`<unk>` 词元，模型完全无法识别这类字符。

#### GPT-4(2023)

分词方法：BPE

词表大小：略多于100000

特殊词元：

· `<|endoftext|>`

·中间填充词元。以下三个词元使LLM能够在考虑前后文的情况下生成补全内容。这种方法在论文“Efficient Training of LanguageModels to Fill in the Middle”中有详细解释，具体细节不在本书的讨论范围。

· `<|fim_prefix|>`

· `<|fim_middle|>`

· `<|fim_suffix|>`

分词后的文本：

<img src="./picture/gpt-4.jpg">


这是一个专注于代码生成的编码器。

GPT-4的分词器行为与其前身GPT-2分词器类似，一些区别如下。

· GPT-4分词器将四个空白字符表示为单个词元。实际上，它甚至为各种长度的空白字符序列（最多83个）都设有特定的词元。

· Python关键字elif在GPT-4中有自己的词元。这一点和前一点都源于模型对代码和自然语言的关注。

· GPT-4分词器使用更少的词元来表示大多数词，例如CAPITALIZATION（用两个词元取代四个词元）和tokens（用一个词元取代三个词元）​。

#### StarCoder2(2024)

 StarCoder2是一个专注于生成代码的150亿参数模型，参见论文“StarCoder2 and TheStack v2: The Next Generation”​。这是对StarCoder的原始工作的延续，参见论文“StarCoder: May the Source be withYou!”​。

分词方法：BPE

词表大小：49152

特殊词元示例：

· `<|endoftext|>`

· 中间填充词元：

· `<fim_prefix>`

· `<fim_middle>`

· `<fim_suffix>`

· `<fim_pad>`

· 在表示代码时，管理上下文很重要。一个文件可能会调用定义在另一个文件中的函数。因此，模型需要某种方式来识别同一代码仓库中位于不同文件中的代码，同时也要区分不同仓库中的代码。这就是为什么StarCoder2使用特殊词元来表示仓库名和文件名：

· `<filename>`

· `<reponame>`

· `<gh_stars>`

分词后的文本：

<img src="./picture/StarCoder2(2024).jpg">

这是一个专注于代码生成的编码器。

  · 类似于GPT-4，它将一系列空白字符编码为单个词元。

· 与我们之前看到的分词方法相比，此方法的主要不同之处是每个数字都被分配了词元（因此600变成了6 0 0）​。这种设计假设这样可以更好地表示数字和数学概念。例如，在GPT-2中，数字870用单个词元表示。但871用两个词元（8 和71）表示。你可以直观地看到，这种表示方式可能会让模型混淆对数字的理解。

#### Galactica

Galactica模型（参见论文“Galactica: ALarge Language Model for Science”​）专注于科学知识领域，是在大量的科学论文、参考资料和知识库上训练而来的。它特别注重分词方式，旨在更好地理解其所表示的数据集的细微差别。例如，它包含了用于表示引用、推理、数学、氨基酸序列和DNA序列的特殊词元。

分词方法：BPE

词表大小：50000

特殊词元：

· `<s>`

· `<pad>`

· `</s>`

· `<unk>`

·引用。引用内容用以下两个特殊词元包裹：

· [START_REF]

 · [END_REF]

下面是论文中的一个使用示例：Recurrentneural networks, long short-term memory[START_REF]Long Short-Term Memory,Hochreiter[END_REF]

·逐步推理：

· `<work>` 是一个有趣的词元，模型用它来进行思维链(chain-of-thought，CoT)推理。

分词后的文本：

<img src="./picture/Galactica.jpg">

Galactica分词器在设计上与StarCoder2类似，都考虑了处理代码的需求。两者对空白字符的编码方式相同：将不同长度的空白字符序列分别编码为单个词元。不同之处在于，Galactica对制表符也进行了同样的处理。因此，在我们见过的所有分词器中，它是唯一一个将由两个制表符组成的字符串("\t\t")分配为单个词元的。

#### Phi-3（和Llama 2）

本书中讨论的Phi-3模型重用了Llama 2的分词器，但添加了一些特殊词元。

分词方法：BPE

词表大小：32000

特殊词元：· `<|endoftext|>`

· 对话词元。随着对话LLM于2023年流行，LLM的对话特性开始成为其主要应用。分词器通过添加表示对话轮次和对话者角色的词元来适应这一趋势。这些特殊词元包括：

· `<|user|>`

· `<|assistant|>`

· `<|system|>`

现在，我们集中回顾一下以上示例：

<img src="./picture/phi-diff.jpg">

### 2.1.6 分词器属性

前文简单讨论了一些训练好的分词器，展示了不同分词器之间的多种差异。是什么决定了它们的分词行为呢？有三大类设计上的选择决定了分词器如何分解文本：分词方法、用于初始化分词器的参数以及训练分词器的目标数据所在的领域。

分词方法

正如我们所见，分词方法有许多种，其中BPE是最流行的一种。每种方法都定义了一种算法，用于选择合适的词元集来表示数据集。要想大概了解这些方法，可以参考HuggingFace的分词器汇总页面。

用于初始化分词器的参数

选择分词方法后，LLM设计者需要设置分词器的一些关键参数，主要包括词表大小、特殊词元和大小写处理策略。

词表大小

分词器的词表中保留多少个词元？​（常见的词表大小的值有30K、50K，但我们越来越多地看到像100K这样更大规模的词表了。​）

特殊词元

我们希望模型跟踪哪些特殊词元？我们可以根据需要添加任意数量的特殊词元，特别是要针对特定用例构建LLM时。常见的特殊词元包括：

· 文本开始词元（例如`<s>`）

· 文本结束词元

· 填充词元

· 未知词元

· CLS词元

· 掩码词元除此之外，LLM设计者还可以添加有助于对所关注问题领域建模的词元，例如Galactica的<work>和[START_REF]词元。

大小写处理策略

在英语等语言中，我们如何处理大小写？是否应该将所有内容转换为小写？​（名称的大小写通常携带有用的信息，但我们是否愿意让那些全大写版本的单词额外占用词表空间？​）

数据领域

即使我们选择相同的方法和参数，分词器的行为也会因其训练所用的数据集而有所不同（这甚至发生在模型训练开始之前）​。前文提到的分词方法通过优化词表来表示特定数据集。从我们的示例中可以看到，这对代码、多语言文本等数据集都有影响。例如在代码方面，我们看到，一个面向文本的分词器可能会对缩进空格进行如下分词（我们用颜色突出显示一些词元）​：

<img src="./picture/def1.jpg">

这对于面向代码的模型来说可能并不是最优方式。通过做出不同的分词选择，面向代码的模型往往可以得到改进：

<img src="./picture/def2.jpg">

这些分词选择能够简化模型的处理过程，从而更有可能提升其性能。

如果想更详细地了解分词器的训练，可以参考Hugging Face上的NLP课程中分词器相关的部分，以及Natural Language Processingwith Transformers, Revised Edition一书。

## 2.2 词元嵌入

现在我们理解了分词，就部分解决了将语言表示给语言模型这一问题。从这个意义上说，语言是词元的序列。如果我们在足够大的词元集上训练一个足够好的模型，它就会开始捕捉训练数据集中出现的复杂模式：

· 如果训练数据包含大量英语文本，通过这些模式，模型就能够表示和生成英语；

· 如果训练数据包含事实性信息（例如维基百科）​，模型就会具备生成一些事实性信息的能力（参见以下说明）​。

解决这个难题的下一步是为这些词元找到最佳的数值表示，使模型能够计算并正确建模文本 中的模式。这些模式呈现给我们的，是模型在特定语言上的连贯性、编码能力或我们期望语言模型具备的任何其他能力。

正如我们在第1章所看到的，这就是嵌入的作用。它们是用于捕捉语言中含义和模式的数值表示空间。

注意：尽管模型的语言连贯性达到了一个较高的水平，事实生成能力也高于平均水平，但这也开始带来新的问题。一些用户开始过度信任模型的事实生成能力（例如，在2023年初，有文章将一些语言模型称为“Google杀手”​）​。但高级用户很快就认识到，仅靠生成模型并不能替代可靠的搜索引擎。于是，RAG（检索增强生成）应运而生，它将搜索和LLM结合起来。我们将在第8章中详细介绍RAG。

### 2.2.1 语言模型为其分词器的词表保存嵌入

分词器经过初始化和训练，就会在其关联的语言模型的训练过程中使用。这就是为什么预训练语言模型与其分词器绑定，在未经训练的情况下不能使用不同的分词器。

如图2-7所示，语言模型为分词器词表中的每个词元都保存了一个嵌入向量。当我们下载预训练语言模型时，模型的一部分就是保存所有这些向量的嵌入矩阵。

<div align="center">
<img src="./picture/2-7.jpg">
<p>图2-7：语言模型为其分词器中的每个词元保存一个与之关联的嵌入向量</p>
</div>


在训练开始之前，这些向量会像模型的其他权重一样被随机初始化，但训练过程会为它们分配值，使其能够执行有意义的行为。

### 2.2.2 使用语言模型创建与上下文相关的词嵌入


我们已经介绍了作为语言模型输入的词元嵌入，接下来让我们看看语言模型如何创建更好的词元嵌入。这是使用语言模型进行文本表示的主要方式之一，为命名实体识别(named-entity recognition，NER)、抽取式文本摘要等应用提供了支持。抽取式文本摘要是指通过突出显示文本中最重要的部分来对长文本进行摘要提炼，而不是生成新的摘要文本。

与使用静态向量表示每个词元或词不同，语言模型会创建与上下文相关(contextualized)的词嵌入（如图2-8所示）​。所谓上下文相关，就是根据词元在上下文中的含义使用不同的表示方式。这些向量可以被其他系统用于各种任务。除了我们在上一段中提到的文本应用外，这些与上下文相关的向量还为DALL· E、Midjourney和Stable Diffusion等AI图像生成系统提供了支持。

让我们看看如何生成与上下文相关的词元嵌入，以下这段代码的大部分内容你现在应该已经很熟悉了：

In [20]:
from transformers import AutoModel, AutoTokenizer
# 加载分词器
tokenizer = AutoTokenizer.from_pretrained("../models/deberta-base")
# 加载语言模型
model = AutoModel.from_pretrained("../models/deberta-v3-xsmall")
# 对句子进行分词
tokens = tokenizer('Hello world', return_tensors='pt')
# 处理词元
output = model(**tokens)[0]

d:\study\python\AiAgent\图解大模型\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: ../models/deberta-v3-xsmall
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 
mask_predictions.classifier.bias           | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from d

我们在这里使用的是DeBERTaV3模型，在撰写本书时，它是最适合生成词元嵌入的语言模型之一，具有体积小、效率高的特点。论文“DeBERTaV3: Improving DeBERTa UsingELECTRA-Style Pre-Training with Gradient-Disentangled Embedding Sharing”详细描述了该模型。

这段代码下载了一个预训练的分词器和模型，并用它们来处理字符串"Hello world"。模型的输出保存在output变量中。我们先打印它的各维度来检查这个变量（我们预计它是一个多维数组）​：

In [21]:
output.shape

torch.Size([1, 4, 384])

跳过第一个维度，我们可以将这个结果理解为4个词元，每个词元都嵌入到一个包含384个值的向量中。第一个维度是批次(batch)维度，要同时向模型发送多个输入句子（如训练时）就要用到它。此时，多个输入句子会被同时处理，从而加快处理速度。

那么，这4个向量是什么？是分词器将两个词拆分成了4个词元，还是发生了其他情况？我们可以运用已经学过的分词器知识来检查它们：

In [22]:
for token in tokens['input_ids'][0]:
    print(tokenizer.decode(token))

[CLS]
Hello
 world
[SEP]


可见，这个分词器和模型会在字符串的开头和结尾分别添加[CLS]和[SEP]词元。

我们的语言模型现在已经处理了文本输入。其输出结果如下：

In [24]:
print(output)

tensor([[[-3.4824,  0.0856, -0.1815,  ..., -0.0614, -0.3911,  0.3030],
         [ 0.1912,  0.3196, -0.2316,  ...,  0.3735,  0.2477,  0.8042],
         [ 0.2076,  0.5024, -0.0485,  ...,  1.2197, -0.2281,  0.8540],
         [-3.4277,  0.0643, -0.1425,  ...,  0.0657, -0.4365,  0.3835]]],
       dtype=torch.float16, grad_fn=<NativeLayerNormBackward0>)


这是语言模型的原始输出。LLM的应用就是建立在这样的输出之上的。

我们在图2-9中回顾了语言模型的输入分词和输出结果。从技术上讲，将词元ID转换为原始嵌入向量是语言模型内部发生的第一步。

<div align="center">
    <img src="./picture/2-9.jpg">
    <p>图2-9：语言模型以原始静态嵌入向量作为输入，并生成与上下文相关的文本嵌入</p>
</div>


这样的可视化对于下一章研究基于Transformer的LLM如何工作至关重要。

## 2.3 文本嵌入（用于句子和整篇文档）

虽然词元嵌入是LLM运作的关键，但许多LLM应用需要处理完整的句子、段落甚至文本文档，这催生了一些特殊的语言模型，它们能够生成文本嵌入——用单个向量来表示长度超过一个词元的文本片段。

我们可以这样理解文本嵌入模型：它接收一段文本，最终生成单个向量，这个向量以某种形式表示该文本并捕捉其含义。图2-10展示了这个过程。

<div align="center">
    <img src="./picture/2-10.jpg">
    <p>图2-10：第一步，我们使用嵌入模型提取特征并将输入文本转换为嵌入向量</p>
</div>

生成文本嵌入有多种方法。最常见的方法之一是对模型生成的所有词元嵌入的值取平均值。然而，高质量的文本嵌入模型往往是专门为文本嵌入任务训练的。

我们可以使用sentence-transformers生成文本嵌入，这是一个流行的利用预训练嵌入模型生成文本嵌入的软件包。与前一章的transformers一样，该软件包可用于加载公开可用的模型。为了演示如何创建嵌入向量，我们使用all-mpnet-base-v2模型。在第4章中，我们将进一步探讨如何为你的任务选择合适的嵌入模型。

In [25]:
from sentence_transformers import SentenceTransformer
# 加载模型
model = SentenceTransformer("../models/all-mpnet-base-v2")
# 将文本转换为文本嵌入
vector = model.encode("Best movie ever!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

嵌入向量的维度取决于底层的嵌入模型。让我们来探索一下模型的维度：

In [26]:
vector.shape

(768,)

现在这个句子被编码成一个维度为768的向量。在本书的第二部分，当我们开始研究应用时，我们将看到这些文本嵌入向量在分类、语义搜索和RAG等各种功能中的巨大作用。

## 2.4 LLM之外的词嵌入

嵌入不仅在文本和语言生成方面有用。事实证明，嵌入（即为对象分配有意义的向量表示）在许多领域都很有用，包括推荐引擎和机器人技术。在本节中，我们将学习如何使用预训练的word2vec嵌入，并简要介绍word2vec是如何创建词嵌入的。了解word2vec的训练方式将为你在第10章学习对比训练做好准备。在下一节中，我们将看到这些嵌入如何用于推荐系统。

### 2.4.1 使用预训练词嵌入

让我们看看如何使用Gensim库下载预训练词嵌入（如word2vec或GloVe）​：

In [28]:
import gensim.downloader as api
# 下载嵌入（66 MB，glove，训练数据来自维基百科，向量大小：50）
# 其他选项包括"word2vec-google-news-300"
# 更多选项请访问gensim-data的GitHub仓库
model = api.load("glove-wiki-gigaword-50")

[==================================================] 100.0% 66.0/66.0MB downloaded


这里，我们下载了在维基百科上训练的大量词的嵌入。然后，我们可以通过查看特定词（例如king）的最近邻来探索嵌入空间：

In [29]:
model.most_similar([model['king']], topn=11)

[('king', 1.0000001192092896),
 ('prince', 0.8236179351806641),
 ('queen', 0.7839043140411377),
 ('ii', 0.7746230363845825),
 ('emperor', 0.7736247777938843),
 ('son', 0.766719400882721),
 ('uncle', 0.7627150416374207),
 ('kingdom', 0.7542161345481873),
 ('throne', 0.7539914846420288),
 ('brother', 0.7492411136627197),
 ('ruler', 0.7434253692626953)]

### 2.4.2 word2vec算法与对比训练

“Efficient Estimation of Word Representations in VectorSpace”一文介绍的word2vec算法，在文章“The Illustrated  Word2vec”中有详细说明。在这里我们将浓缩其核心思想，因为在下一节讨论推荐引擎的嵌入创建方法时会以此为基础。

与LLM一样，word2vec也是在从文本生成的样本上进行训练的。假设我们有Frank Herbert的小说Dune中的一段文本：​“Thou shalt not make a machine in the likeness of ahuman mind”​。该算法使用滑动窗口来生成训练样本。比如，我们可以设置窗口大小为2，即考虑中心词两侧各两个相邻词。

嵌入向量是通过分类任务生成的。这种任务用于训练神经网络，以预测词是否经常出现在相同的上下文中（这里的上下文指的是我们建模的训练数据集中的多个句子）​。我们可以将其理解为一个神经网络，它接收两个词作为输入，如果这两个词倾向于出现在相同的上下文中则输出1，否则输出0。

在滑动窗口的第一个位置，我们可以生成四个训练样本，如图2-11所示。

<div align="center">
    <img src="./picture/2-11.jpg">
    <p>图2-11：滑动窗口用于生成word2vec算法的训练样本，以便后续预测两个词是否为相邻词</p>
</div>

在每个生成的训练样本中，中心词作为第一个输入，在各个训练样本中，其相邻词分别作为第二个输入。我们期望最终训练好的模型能够对这种相邻关系进行分类，当接收到的两个输入词是相邻词时输出1。这些训练样本如图2-12所示。

<div align="center">
    <img src="./picture/2-12.jpg">
    <p>图2-12：每个生成的训练样本展示了一对相邻词</p>
</div>

然而，如果我们的数据集中只有目标值为1的样本（正例）​，那么模型可能会投机取巧，通过一直输出1来获得完美表现。为解决这个问题，我们需要用非典型相邻词的样本来丰富训练数据集。这些被称为负例，如图2-13所示。

<div align="center">
    <img src="./picture/2-13.jpg">
    <p>图2-13：我们需要向模型展示负例，即通常不相邻的词。更好的模型能够更好地区分正例和负例</p>
</div>

事实证明，在选择负例时并不需要过于严谨。仅仅是借助从随机生成的样本中检测正例的能力，就能得到很多有用的模型[这受到了噪声对比估计(noise-contrastive estimation)这一重要概念的启发，详见论文“Noise-Contrastive Estimation: ANew Estimation Principle for Unnormalized StatisticalModels”​]​。因此在这种情况下，我们随机获取一些词，将它们添加到数据集中，并将其标注为非相邻词（模型看到它们时应该输出0）​。

至此，我们已经了解了word2vec的两个主要概念（见图2-14）​：skip-gram，选择相邻词的方法；负采样(negativesampling)，通过从数据集中随机采样来添加负例。

<div align="center">
    <img src="./picture/2-14.jpg">
    <p>图2-14：skip-gram和负采样是word2vec算法背后的两个主要思想，它们在许多可以表述为词元序列问题的问题中很有用</p>
</div>

我们可以从连续文本中生成数百万甚至数十亿个这样的训练样本。在开始用这个数据集训练神经网络之前，我们需要做一些分词决策，就像我们在LLM分词器中看到的那样，包括如何处理大小写和标点符号，以及词表中包含多少词元。

   然后，我们为每个词元创建一个嵌入向量并随机初始化，如图2-15所示。在实践中，这是一个矩阵，其维度为：词表大小×嵌入向量的维度。

<div align="center">
    <img src="./picture/2-15.jpg">
    <p>图2-15：词表中的词及其起始的、随机的、未初始化的嵌入向量</p>
</div>

接下来，我们在每个样本上训练模型，输入两个嵌入向量并预测它们是否相邻。这个过程如图2-16所示。

<div align="center">
    <img src="./picture/2-16.jpg">
    <p>图2-16：训练神经网络来预测两个词是否相邻。它在训练过程中不断优化嵌入向量，最终生成训练好的嵌入表示</p>
</div>

根据模型预测正确与否，典型的机器学习训练步骤会调整嵌入向量，以便模型在下次遇到这两个向量时，更准确地进行预测。在训练过程结束时，词表中的所有词元获得了更好的词嵌入表示。

这种接收两个向量并预测它们是否具有某种关系的模型思想，是机器学习中最强大的思想之一，并且在语言模型中屡试不爽。这就是为什么我们要在第10章专门讨论这个概念，以及它如何优化语言模型来完成特定任务（如句子嵌入和检索）​。

这个思想也是连接文本和图像等不同模态的核心，这对AI图像生成模型来说至关重要，我们将在第9章详细讨论多模态模型。在多模态形式中，模型会接收一张图片和一段描述文本，然后预测该文本是否描述了这张图片。

## 2.5 推荐系统中的嵌入

正如我们提到的，嵌入的概念在许多其他领域都很有用。在工业界，它被广泛应用于推荐系统。

### 2.5.1 基于嵌入的歌曲推荐

在本节中，我们将使用word2vec算法，利用人工创建的音乐播放列表来嵌入歌曲。想象一下，我们把每首歌曲都当作一个词或词元来处理，把每个播放列表当作一个句子，这些嵌入就可以用来推荐经常出现在同一个播放列表中的歌曲。

我们将使用的Playlist数据集是由康奈尔大学的Shuo Chen收集的。它包含了来自美国数百个广播电台的播放列表。图2-17展示了这个数据集的形式。

<div align="center">
    <img src="./picture/2-17.jpg">
    <p>图2-17：为了获得捕捉歌曲相似性的歌曲嵌入，我们将使用由一系列播放列表组成的数据集，每个播放列表包含一组歌曲</p>
<div>

在深入了解其构建方式之前，让我们先演示一下最终的效果。我们输入几首歌曲，看看它会推荐什么。

我们从Michael Jackson的“Billie Jean”​（歌曲ID为3822）开始：


print_recommendations(3822)

<img src="./picture/3822.jpg">

看起来很合理，Madonna、Prince的歌曲，以及MichaelJackson的其他歌曲是最近邻。

让我们从流行乐转向说唱领域，看看2Pac的“CaliforniaLove”的近邻：

print_recommendations(842)

<img src="./picture/842.jpg">

这个列表看起来也相当合理！现在我们已经看到它能工作，让我们看看如何构建这样一个系统。

### 2.5.2 训练歌曲嵌入模型

首先加载包含歌曲播放列表的数据集，以及每首歌曲的元数据，如标题和艺术家：

In [35]:
import pandas as pd

# 读取播放列表训练集
with open('train.txt', 'r', encoding='utf-8') as f:
    data = f.read()
    lines = data.split('\n')[2:]   # 跳过前两行元数据

# 删除只有一首歌的播放列表
playlists = [s.rstrip().split() for s in lines if len(s.split()) > 1]

# 读取歌曲元数据（ID \t Title \t Artist）
with open('song_hash.txt', 'r', encoding='utf-8') as f:
    song_lines = f.read().splitlines()

# 每行按 \t 切成三列
songs = [line.split('\t') for line in song_lines if line.strip()]

# 构建 DataFrame
songs_df = pd.DataFrame(data=songs, columns=['id', 'title', 'artist'])
songs_df = songs_df.set_index('id')

print(songs_df.head())
print("歌曲总数：", len(songs_df))
print("播放列表数：", len(playlists))

                                                title      artist
id                                                               
0                        Gucci Time (w\/ Swizz Beatz)  Gucci Mane
1   Aston Martin Music (w\/ Drake & Chrisette Mich...   Rick Ross
2                       Get Back Up (w\/ Chris Brown)        T.I.
3                  Hot Toddy (w\/ Jay-Z & Ester Dean)       Usher
4                                        Whip My Hair      Willow
歌曲总数： 75262
播放列表数： 11088


现在我们已经保存了，让我们检查一下列表playlists。其中每个元素都是一个包含一系列歌曲ID的播放列表：

In [36]:
print('Playlist #1:\n', playlists[0])
print('Playlist #2:\n', playlists[1])

Playlist #1:
 ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '2', '42', '43', '44', '45', '46', '47', '48', '20', '49', '8', '50', '51', '52', '53', '54', '55', '56', '57', '25', '58', '59', '60', '61', '62', '3', '63', '64', '65', '66', '46', '47', '67', '2', '48', '68', '69', '70', '57', '50', '71', '72', '53', '73', '25', '74', '59', '20', '46', '75', '76', '77', '59', '20', '43']
Playlist #2:
 ['78', '79', '80', '3', '62', '81', '14', '82', '48', '83', '84', '17', '85', '86', '87', '88', '74', '89', '90', '91', '4', '73', '62', '92', '17', '53', '59', '93', '94', '51', '50', '27', '95', '48', '96', '97', '98', '99', '100', '57', '101', '102', '25', '103', '3', '104', '105', '106', '107', '47', '108', '109', '110', '111', '112', '113', '25', '63', '62', '114', '115', '84', '116', '117', '11

让我们训练这个模型：

In [37]:
from gensim.models import Word2Vec
# 训练我们的word2vec模型
model = Word2Vec(
    playlists, vector_size=32, window=20, negative=50, min_count=1, workers=4
)

训练需要一两分钟的时间，最终会为每首歌曲计算出嵌入向量。现在我们可以像之前处理词语一样，使用这些嵌入向量来寻找相似的歌曲：

In [38]:
song_id = 2172
# 让模型找出与歌曲2172相似的歌曲
model.wv.most_similar(positive=str(song_id))

[('3167', 0.9984921813011169),
 ('2976', 0.9971234202384949),
 ('2849', 0.9968960285186768),
 ('1922', 0.9968212246894836),
 ('3105', 0.9967883825302124),
 ('3094', 0.9962696433067322),
 ('11473', 0.9961373209953308),
 ('5586', 0.9958821535110474),
 ('10084', 0.9958798289299011),
 ('3116', 0.9958036541938782)]

这是与歌曲2172具有最相似的嵌入表示的歌曲列表。

在这个例子中，这首歌是：

In [39]:
print(songs_df.iloc[2172])

title     Fade To Black
artist        Metallica
Name: 2172 , dtype: str


推荐给出的都属于重金属和硬摇滚风格的歌曲：

In [40]:
import numpy as np
def print_recommendations(song_id):
    similar_songs = np.array(
        model.wv.most_similar(positive=str(song_id),topn=5)
    )[:,0]
    return songs_df.iloc[similar_songs]
# 提取推荐结果
print_recommendations(2172)

,title,artist
id,,
3167,Unchained,Van Halen
2976,I Don't Know,Ozzy Osbourne
2849,Run To The Hills,Iron Maiden
1922,One,Metallica
3105,Working Man,Rush


## 2.6 小结

在本章中，我们介绍了LLM词元、分词器以及使用词元嵌入的实用方法。这为我们下一章深入研究语言模型做好了准备，同时也为我们学习如何在语言模型之外使用嵌入打开了大门。

我们探讨了分词器如何作为处理LLM输入的第一步，将原始文本输入转换为词元ID。常见的分词方案包括将文本分解为词、子词、字符或字节，具体取决于特定应用的要求。

通过对现实世界预训练分词器（从BERT到GPT-2、GPT-4和其他模型）的探索，我们了解了某些分词器在某些方面表现更好（例如，保留大小写、换行符或其他语言的词元等信息）​；而在其他方面，分词器之间仅存在差异（例如，它们如何分解某些词）​，并无优劣之分。

分词器设计中有三个主要决策点：分词器算法（如BPE、WordPiece、SentencePiece）​、分词参数（包括词表大小、特殊词元、大小写处理策略和不同语言的处理）以及用于训练分词器的数据集。

语言模型能够生成高质量与上下文相关的词元嵌入，这种嵌入改进了原始的静态嵌入。这些与上下文相关的词元嵌入可以用于命名实体识别、抽取式文本摘要和文本分类等任务。除了生成词元嵌入，语言模型还可以生成涵盖整个句子甚至文档的文本嵌入。这为本书第二部分将要展示的众多语言模型应用提供了强大支持。

在LLM之前，word2vec、GloVe和fastText等词嵌入方法非常流行。在语言处理中，这些方法已经在很大程度上被语言模型产生的与上下文相关的词嵌入所取代。word2vec算法依赖两个主要思想：skip-gram模型和负采样。它还使用了与我们将在第10章看到的类似的对比训练方法。

从根据播放列表构建音乐推荐系统的例子中我们看到，嵌入对于创建和改进推荐系统非常有用。

在下一章中，我们将深入探讨分词后的处理过程：LLM如何处理这些词元并生成文本？我们将探讨使用Transformer架构的LLM的主要工作原理。